In [ ]:
import random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
SR       = 32000
DURATION = 5
N_MFCC   = 20
N_FEAT   = N_MFCC * 4 + 2
SEED     = 42

BASE_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE_DIR / 'train_audio'
TRAIN_SC    = BASE_DIR / 'train_soundscapes'
META_CSV    = BASE_DIR / 'train.csv'
TAX_CSV     = BASE_DIR / 'taxonomy.csv'
SC_CSV      = BASE_DIR / 'train_soundscapes_labels.csv'

FEAT_CACHE = Path('/kaggle/working/features_v3.npz')

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
df  = pd.read_csv(META_CSV)
tax = pd.read_csv(TAX_CSV)
sc  = pd.read_csv(SC_CSV)

species_list   = tax['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}
NUM_CLASSES    = len(species_list)

labels   = df['primary_label'].astype(str)
counts   = labels.value_counts()
valid_df = df[labels.isin(counts[counts >= 2].index)].reset_index(drop=True)

# fix train/val split here so build_features and val_setup use the same indices
sc_val_df   = sc.sample(frac=0.2, random_state=SEED)
sc_train_df = sc.drop(sc_val_df.index)

print(f'Species to predict  : {NUM_CLASSES}')
print(f'Training clips      : {len(valid_df)}')
print(f'Soundscape train    : {len(sc_train_df)}')
print(f'Soundscape val      : {len(sc_val_df)}')

## Feature Extraction

20 MFCCs + 20 delta MFCCs (temporal dynamics) x (mean + std) + centroid + ZCR = **82 features**.

In [ ]:
def extract_features(audio):
    target = SR * DURATION
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    audio = audio[:target]

    mfcc  = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    cent  = librosa.feature.spectral_centroid(y=audio, sr=SR)
    zcr   = librosa.feature.zero_crossing_rate(y=audio)

    return np.concatenate([
        mfcc.mean(axis=1),    # 20  spectral envelope
        mfcc.std(axis=1),     # 20  envelope variability
        delta.mean(axis=1),   # 20  how spectrum changes over time
        delta.std(axis=1),    # 20  variability of change
        cent.mean(axis=1),    #  1  brightness
        zcr.mean(axis=1),     #  1  noisiness
    ]).astype(np.float32)

In [ ]:
def _sc_row_to_target(row):
    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    for sp in str(row['primary_label']).split(';'):
        sp = sp.strip()
        if sp in species_to_idx:
            target[species_to_idx[sp]] = 1.0
    return target

In [ ]:
_from_cache = FEAT_CACHE.exists()

if _from_cache:
    print('Loading cached features...')
    data       = np.load(FEAT_CACHE)
    X_clips    = data['X_clips']
    y_clips    = data['y_clips']
    X_sc_all   = data['X_sc_all']
    Y_sc_all   = data['Y_sc_all']
    print(f'  Clips : {X_clips.shape}')
    print(f'  SC    : {X_sc_all.shape}')
else:
    t0 = time.time()
    print(f'Extracting features from {len(valid_df)} clips...')
    X_rows, y_rows = [], []
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc='Clips'):
        fp = TRAIN_AUDIO / row['filename']
        try:
            total    = librosa.get_duration(path=str(fp))
            offset   = max(0.0, (total - DURATION) / 2.0)
            audio, _ = librosa.load(str(fp), sr=SR, offset=offset, duration=DURATION)
            if np.abs(audio).max() < 1e-4:
                continue
            X_rows.append(extract_features(audio))
            y_rows.append(species_to_idx.get(str(row['primary_label']), -1))
        except Exception:
            continue
    X_clips = np.array(X_rows, dtype=np.float32)
    y_clips = np.array(y_rows, dtype=np.int32)
    mask    = y_clips >= 0
    X_clips, y_clips = X_clips[mask], y_clips[mask]
    print(f'Clips done: {len(X_clips)} in {time.time() - t0:.0f}s')

In [ ]:
if not _from_cache:
    t0 = time.time()
    print(f'Extracting features from {len(sc)} soundscape windows...')
    X_sc_rows, Y_sc_rows = [], []
    for _, row in tqdm(sc.iterrows(), total=len(sc), desc='Soundscapes'):
        parts     = str(row['start']).split(':')
        start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        try:
            audio, _ = librosa.load(str(TRAIN_SC / row['filename']),
                                    sr=SR, offset=float(start_sec), duration=DURATION)
            X_sc_rows.append(extract_features(audio))
            Y_sc_rows.append(_sc_row_to_target(row))
        except Exception:
            X_sc_rows.append(np.zeros(N_FEAT, dtype=np.float32))
            Y_sc_rows.append(np.zeros(NUM_CLASSES, dtype=np.float32))

    X_sc_all = np.array(X_sc_rows, dtype=np.float32)
    Y_sc_all = np.array(Y_sc_rows, dtype=np.float32)

    np.savez(FEAT_CACHE, X_clips=X_clips, y_clips=y_clips,
             X_sc_all=X_sc_all, Y_sc_all=Y_sc_all)
    print(f'Soundscapes done: {X_sc_all.shape} in {time.time() - t0:.0f}s')

In [ ]:
val_pos   = sc_val_df.index.tolist()
train_pos = sc_train_df.index.tolist()

X_sc_val,   Y_sc_val   = X_sc_all[val_pos],   Y_sc_all[val_pos]
X_sc_train, Y_sc_train = X_sc_all[train_pos], Y_sc_all[train_pos]

Y_clips = np.eye(NUM_CLASSES, dtype=np.float32)[y_clips]

X_train = np.vstack([X_clips, X_sc_train])
Y_train = np.vstack([Y_clips,  Y_sc_train])

print(f'X_train : {X_train.shape}  (clips + soundscape train windows)')
print(f'X_val   : {X_sc_val.shape}')

## Classifier Comparison

Three classifiers, Binary Relevance (one classifier per species), evaluated on held-out soundscape windows.

In [ ]:
# scale once - LR and SVM need it, RF does not
scaler    = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_sc_val)

Y_val  = Y_sc_val
active = Y_val.sum(axis=1)
print(f'Val windows          : {len(Y_val)}')
print(f'Avg species / window : {active.mean():.1f}  (max {int(active.max())})')

# species with at least one positive example in training - avoids single-class column error
active_cols = np.where(Y_train.max(axis=0) > 0)[0]
print(f'Species with training positives : {len(active_cols)} / {NUM_CLASSES}')

In [ ]:
class _FullPredictor:
    """Wraps a MultiOutputClassifier trained on active_cols, padding zeros for inactive species."""
    def __init__(self, clf, cols, n_total):
        self._clf = clf
        self._cols = cols
        self._n = n_total

    def predict_proba(self, X):
        sub = self._clf.predict_proba(X)
        out = [np.column_stack([np.ones(len(X)), np.zeros(len(X))]) for _ in range(self._n)]
        for i, p in zip(self._cols, sub):
            out[i] = p
        return out


def macro_auc(clf, X, Y):
    P_list = clf.predict_proba(X)
    P = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
    for i, p in enumerate(P_list):
        if p.shape[1] == 2:
            P[:, i] = p[:, 1]
    aucs = [
        roc_auc_score(Y[:, i], P[:, i])
        for i in range(NUM_CLASSES)
        if 0 < Y[:, i].sum() < len(Y)
    ]
    return round(float(np.mean(aucs)), 4) if aucs else float('nan')

In [ ]:
results = {}

In [ ]:
print('Training Logistic Regression  (Binary Relevance, 234 classifiers)...')
t0 = time.time()

_lr_inner = MultiOutputClassifier(
    LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED),
    n_jobs=-1
)
_lr_inner.fit(X_train_s, Y_train[:, active_cols])
lr = _FullPredictor(_lr_inner, active_cols, NUM_CLASSES)
train_time = round(time.time() - t0, 1)
auc = macro_auc(lr, X_val_s, Y_val)

results['Logistic Regression'] = {'Val AUC (macro)': auc, 'Train time (s)': train_time}
print(f'  Train time : {train_time}s')
print(f'  Val AUC    : {auc}')

In [ ]:
print('Training Random Forest  (native multi-output, 100 trees)...')
t0 = time.time()

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)
rf.fit(X_train, Y_train)
train_time = round(time.time() - t0, 1)
auc = macro_auc(rf, X_sc_val, Y_val)

results['Random Forest'] = {'Val AUC (macro)': auc, 'Train time (s)': train_time}

feat_names = (
    [f'MFCC{i}_mean'  for i in range(N_MFCC)] +
    [f'MFCC{i}_std'   for i in range(N_MFCC)] +
    [f'dMFCC{i}_mean' for i in range(N_MFCC)] +
    [f'dMFCC{i}_std'  for i in range(N_MFCC)] +
    ['centroid_mean', 'zcr_mean']
)
top5 = rf.feature_importances_.argsort()[::-1][:5]
print(f'  Train time  : {train_time}s')
print(f'  Val AUC     : {auc}')
print('  Top-5 features:')
for idx in top5:
    print(f'    {feat_names[idx]:20s}  {rf.feature_importances_[idx]:.4f}')

In [ ]:
print('Training Linear SVM  (Binary Relevance + probability calibration)...')
t0 = time.time()

_svm_inner = MultiOutputClassifier(
    CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=SEED)),
    n_jobs=-1
)
_svm_inner.fit(X_train_s, Y_train[:, active_cols])
svm = _FullPredictor(_svm_inner, active_cols, NUM_CLASSES)
train_time = round(time.time() - t0, 1)
auc = macro_auc(svm, X_val_s, Y_val)

results['Linear SVM'] = {'Val AUC (macro)': auc, 'Train time (s)': train_time}
print(f'  Train time : {train_time}s')
print(f'  Val AUC    : {auc}')

In [ ]:
df_results = pd.DataFrame(results).T.rename_axis('Classifier')
df_results = df_results.sort_values('Val AUC (macro)', ascending=False)

print(df_results.to_string())
print()
if df_results['Val AUC (macro)'].notna().any():
    best = df_results['Val AUC (macro)'].idxmax()
    print(f'Best: {best}  (AUC {df_results.loc[best, "Val AUC (macro)"]:.4f})')
else:
    print('No valid AUC scores - check that the feature cache was built with full data.')